In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/raw/telco_churn.csv")
print(df.shape)
df.head()

(5043, 22)


,Unnamed: 0,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,7590-VHVEG,Female,False,True,False,1,False,NaN,DSL,False,True,False,False,False,False,Month-to-month,True,Electronic check,29.850000,29.850000381469727,False
1,1,5575-GNVDE,Male,False,False,False,34,True,False,DSL,True,False,True,False,False,False,One year,False,Mailed check,56.950001,1889.5,False
2,2,3668-QPYBK,Male,False,False,False,2,True,False,DSL,True,True,False,False,False,False,Month-to-month,True,Mailed check,53.849998,108.1500015258789,True
3,3,7795-CFOCW,Male,False,False,False,45,False,NaN,DSL,True,False,True,True,False,False,One year,False,Bank transfer (automatic),42.299999,1840.75,False
4,4,9237-HQITU,Female,False,False,False,2,True,False,Fiber optic,False,False,False,False,False,False,Month-to-month,True,Electronic check,70.699997,151.64999389648438,True


In [2]:
df = df.drop(columns=["Unnamed: 0"])
print(df.shape)
df.columns.tolist()

(5043, 21)


['customerID',
 'gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'tenure',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod',
 'MonthlyCharges',
 'TotalCharges',
 'Churn']

In [3]:
# 1) juntar os quatro valores em apenas dois: No / Yes
df["Churn"] = df["Churn"].replace({"False": "No", "True": "Yes"})

# 2) ver como ficou (ainda mostrando ausentes)
print(df["Churn"].value_counts(dropna=False))

Churn
No     3706
Yes    1336
NaN       1
Name: count, dtype: int64


In [4]:
# guardar quantas linhas havia antes, para confirmar a remoção
antes = df.shape[0]

# remover linhas onde Churn está ausente
df = df.dropna(subset=["Churn"])

print("Linhas antes:", antes)
print("Linhas depois:", df.shape[0])
print(df["Churn"].value_counts(dropna=False))

Linhas antes: 5043
Linhas depois: 5042
Churn
No     3706
Yes    1336
Name: count, dtype: int64


In [5]:
# ver o tipo atual (confirmar que ainda é texto)
print("Tipo antes:", df["TotalCharges"].dtype)

# converter para número; o que não for convertível vira NaN (ausente)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("Tipo depois:", df["TotalCharges"].dtype)
print("Ausentes criados:", df["TotalCharges"].isna().sum())

Tipo antes: str
Tipo depois: float64
Ausentes criados: 8


In [6]:
# olhar as linhas onde TotalCharges ficou ausente
df[df["TotalCharges"].isna()][["tenure", "MonthlyCharges", "TotalCharges"]]

,tenure,MonthlyCharges,TotalCharges
488,0,52.549999,NaN
753,0,20.250000,NaN
936,0,80.849998,NaN
1082,0,25.750000,NaN
1340,0,56.049999,NaN
3218,0,19.700000,NaN
4670,0,73.350000,NaN
4754,0,61.900000,NaN


In [7]:
# preencher os TotalCharges ausentes com 0 (clientes com tenure = 0)
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# confirmar que já não há ausentes nesta coluna
print("Ausentes em TotalCharges:", df["TotalCharges"].isna().sum())

Ausentes em TotalCharges: 0


In [8]:
# colunas de serviços onde "vazio" significa "o cliente não tem esse serviço"
colunas_servico = [
    "MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies"
]

# substituir os ausentes por uma categoria explícita
df[colunas_servico] = df[colunas_servico].fillna("No service")

# confirmar que já não há ausentes em nenhuma coluna do dataset
print(df.isna().sum())

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


In [9]:
import os

os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/telco_churn_clean.csv", index=False)

print("Guardado. Formato final:", df.shape)

Guardado. Formato final: (5042, 21)
